# Manhattan Reasoning Gym — build a design & run it on a real FPGA

This tutorial takes one hardware design from **local feedback** to **real
silicon**, using the `mrg` SDK.

The flow:

1. write a small hardware design (`design.py`),
2. get fast **synth / place-and-route** feedback locally (`mrg.build`) — no board,
3. **program a real FPGA** and talk to it over the bus (`mrg.cloud`).

**Prerequisites**
- Python 3.11+
- **Docker** — `mrg.build` runs the pinned toolchain image for you, so you never
  install yosys/nextpnr/LiteX. (Until the image is on a registry, build it once:
  `docker pull ghcr.io/barnard-pl-labs/mrg-sandbox:latest`)
- An **API key** for the cloud step (`mrg login` or `MRG_API_KEY`).

## 1. Install the SDK

In [ ]:
# Locate the repo root (works from anywhere inside the checkout).
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
print("repo root:", ROOT)

In [ ]:
# Install the mrg SDK (one package). Install the SDK from this checkout.
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)],
               check=True)
print("installed manhattan-reasoning-gym")

## 2. The design

We'll use the example **multiply-accumulate** design: it exposes the Wishbone
register contract every user design implements. Register map (word addresses):
`0`=write A, `1`=write B (does `acc += A*B`), `2`=read `acc`, `3`=reset.

In [ ]:
DESIGN = ROOT / "examples" / "design.py"
print(DESIGN.read_text())

## 3. Local build feedback — `mrg.build`

`mrg.build.synth` / `pnr` return a structured report. On your machine they
transparently run the pinned Docker image (no toolchain to install); inside the
sandbox image they run in-process. Either way you get a `BuildReport`.

In [ ]:
import manhattan_reasoning_gym as mrg

synth = mrg.build.synth(str(DESIGN))          # fast: resource utilization
print("synth  ->", {"dsp": synth.util.dsp.used, "ff": synth.util.ff.used,
                    "bram": synth.util.bram.used})

pnr = mrg.build.pnr(str(DESIGN))              # full SoC: real Fmax + timing
print(f"pnr    -> fits={pnr.fits}  Fmax={pnr.fmax_mhz:.1f} MHz  "
      f"timing_met={pnr.timing_met}")
print("util % ->", {k: getattr(pnr.util, k).pct
                    for k in ("logic", "ff", "bram", "dsp")})

Tweak the design (e.g. widen `WIDTH` in `design.py`) and re-run to watch the DSP
count and Fmax move. This loop is fast, free, and needs no board.

## 4. Run it on a real FPGA — `mrg.cloud`

When you're happy, program a real ECP5 and talk to it over the bus. This needs
an API key (`mrg login` or `MRG_API_KEY`), so the cell below runs for real only
if a key is present — otherwise it prints the code.

In [ ]:
import os

# In a notebook we drive the board directly. (The script equivalent is
# `@app.local_entrypoint()` + `mrg run` — see examples/hello_wishbone.)
# `with app:` programs the FPGA on first read/write and releases it on exit,
# even if a call raises.
if os.environ.get("MRG_API_KEY"):
    app = mrg.cloud.App("tutorial", design=str(DESIGN))
    with app:
        app.write(0x0, 3)                 # A = 3   (builds + flashes)
        app.write(0x4, 4)                 # acc += 3 * 4
        print("acc =", app.read(0x8))     # expect 12
else:
    print("No MRG_API_KEY set \u2014 here's the code you'd run on a real board:\n")
    print('''app = mrg.cloud.App("tutorial", design="design.py")
with app:                          # programs on first use, releases on exit
    app.write(0x0, 3)     # A = 3
    app.write(0x4, 4)     # acc += 3 * 4
    print("acc =", app.read(0x8))    # -> 12''')

### Recap

- **`mrg.build`** — local synth/pnr feedback, no board, just Docker.
- **`mrg.cloud`** — program + drive a real FPGA with an API key.

Next: [`02_sandboxing.ipynb`](02_sandboxing.ipynb) — run an **agent** in a
locked sandbox that promotes designs to silicon through a broker.